In [ ]:
import sys
# Run before importing LangChain. Restart the kernel after changing installed versions.
!{sys.executable} -m pip install --upgrade langchain==0.3.26 langchain-community==0.3.26 langchain-core==0.3.86 langchain-openai==0.3.24 openai==1.109.1 httpx==0.28.1 tavily-python python-dotenv requests certifi

In [ ]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")


In [ ]:
from langchain.agents import create_react_agent, AgentExecutor

In [ ]:
search_tool = TavilySearchResults(max_results=2)

In [ ]:
@tool
def get_weather_data(city: str) -> str:
    """Fetch current weather information for a city."""
    if not WEATHERSTACK_API_KEY:
        return "Weather lookup is unavailable: set WEATHERSTACK_API_KEY in .env."

    try:
        response = requests.get(
            "https://api.weatherstack.com/current",
            params={"access_key": WEATHERSTACK_API_KEY, "query": city},
            timeout=15,
        )
        response.raise_for_status()
        data = response.json()
    except (requests.RequestException, ValueError) as exc:
        return f"Could not fetch weather data for {city}: {exc}"

    current = data.get("current")
    if not current:
        message = data.get("error", {}).get("info", "No weather data returned")
        return f"Could not fetch weather data for {city}: {message}"

    description = current.get("weather_descriptions", ["Unknown"])[0]
    return (
        f"City: {city}\n"
        f"Temperature: {current.get('temperature', 'Unknown')} {chr(176)}C\n"
        f"Weather: {description}\n"
        f"Humidity: {current.get('humidity', 'Unknown')}%"
    )

In [ ]:
result = search_tool.invoke("Give me the latest news on AI")
result

In [ ]:
if not OPENAI_API_KEY:
    raise ValueError("Set OPENAI_API_KEY in the project .env file before using the model.")

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY,
    default_headers={"Accept-Encoding": "gzip, deflate"},
)

In [33]:
response = llm.invoke("Tell me a joke about AI")
response

OpenAIConnectionError: Connection error.

In [ ]:
react_prompt = """Answer the following question using the available tools.

Available tools:
{tools}

Use this format:
Question: the question to answer
Thought: what to do next
Action: one of [{tool_names}]
Action Input: the tool input
Observation: the tool result
... repeat Thought/Action/Action Input/Observation as needed ...
Thought: I know the final answer
Final Answer: the answer to the question

Question: {input}
Thought:{agent_scratchpad}"""
prompt = PromptTemplate.from_template(react_prompt)

In [ ]:
prompt

In [ ]:
tools = [search_tool, get_weather_data]

In [ ]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [ ]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [ ]:
response = agent_executor.invoke({
    "input": "Find the capital of India and then find its current weather."
})

In [ ]:
print(response["output"])